In [1]:
import os
import pandas as pd
import numpy as np

print(os.listdir('.'))

['.config', 'suvidha_loan_predictions_raw - suvidha_loan_predictions_raw.csv.csv', 'sample_data']


In [2]:
import numpy as np
import pandas as pd

np.random.seed(2026)

n_loans = 2400

loan_ids = [f"LN-{1000 + i}" for i in range(n_loans)]

# Default rate ~9%
# Generate actual_default boolean array first
# ~2338 valid, 42 blank rows
is_blank_actual = np.random.choice([True, False], size=n_loans, p=[40/2400, 2360/2400])

# Among valid rows, ~9% actual defaults
defaults_prob = 0.09
actual_defaults_bool = np.random.binomial(1, defaults_prob, size=n_loans)

# Risk score generation:
# Defaults tend to have higher risk_scores, but model flags low threshold (e.g. 0.35)
risk_score = np.where(
    actual_defaults_bool == 1,
    np.random.beta(5, 2, size=n_loans), # higher scores for actual defaults
    np.random.beta(1.5, 4, size=n_loans) # lower scores for non-defaults, but some moderate/high
)
risk_score = np.round(risk_score, 4)

# Model flag: default model uses cutoff around 0.30 to ensure high recall
model_flag_bool = (risk_score >= 0.32).astype(int)

# Introduce 22 blank rows for model_flag (independent or slightly overlapping)
is_blank_model = np.random.choice([True, False], size=n_loans, p=[22/2400, 2378/2400])

# Encodings for actual_default and model_flag
yes_encodings = ['Yes', 'Y', '1', 'true', 'TRUE', 'yes']
no_encodings = ['No', 'N', '0', 'false', 'FALSE', 'no']

raw_actual_default = []
for i in range(n_loans):
    if is_blank_actual[i]:
        raw_actual_default.append("")
    else:
        if actual_defaults_bool[i] == 1:
            raw_actual_default.append(np.random.choice(yes_encodings))
        else:
            raw_actual_default.append(np.random.choice(no_encodings))

raw_model_flag = []
for i in range(n_loans):
    if is_blank_model[i]:
        raw_model_flag.append("")
    else:
        if model_flag_bool[i] == 1:
            raw_model_flag.append(np.random.choice(yes_encodings))
        else:
            raw_model_flag.append(np.random.choice(no_encodings))

# Loan amount, tenure, interest rate
# Defaulted loans vs Non-defaulted loans
# Let's give defaulted loans slightly higher interest rate and slightly higher loan amount
loan_amounts = np.where(
    actual_defaults_bool == 1,
    np.random.normal(88000, 25000, size=n_loans),
    np.random.normal(72000, 22000, size=n_loans)
)
loan_amounts = np.clip(loan_amounts, 10000, 250000).round(-2)

raw_loan_amounts = []
for i in range(n_loans):
    amt = int(loan_amounts[i])
    if np.random.rand() < 0.05: # "Rs 85,000" text
        raw_loan_amounts.append(f"Rs {amt:,}")
    else:
        raw_loan_amounts.append(amt)

tenure_months = np.random.choice([12, 18, 24, 36, 48], size=n_loans, p=[0.2, 0.3, 0.3, 0.1, 0.1])

interest_rates = np.where(
    actual_defaults_bool == 1,
    np.random.normal(18.5, 2.5, size=n_loans),
    np.random.normal(15.2, 2.0, size=n_loans)
)
interest_rates = np.clip(interest_rates, 8.0, 28.0).round(2)

borrower_age = np.random.randint(21, 62, size=n_loans)
monthly_income = np.random.normal(35000, 12000, size=n_loans).clip(10000, 120000).round(-2)

raw_borrower_age = []
raw_monthly_income = []

for i in range(n_loans):
    r_age = np.random.rand()
    if r_age < 0.02:
        raw_borrower_age.append("")
    elif r_age < 0.04:
        raw_borrower_age.append(-1)
    else:
        raw_borrower_age.append(borrower_age[i])

    r_inc = np.random.rand()
    if r_inc < 0.02:
        raw_monthly_income.append("")
    elif r_inc < 0.03:
        raw_monthly_income.append("-1")
    elif r_inc < 0.04:
        raw_monthly_income.append("not disclosed")
    else:
        raw_monthly_income.append(monthly_income[i])

df_suvidha_raw = pd.DataFrame({
    'loan_id': loan_ids,
    'loan_amount': raw_loan_amounts,
    'tenure_months': tenure_months,
    'interest_rate': interest_rates,
    'borrower_age': raw_borrower_age,
    'monthly_income': raw_monthly_income,
    'actual_default': raw_actual_default,
    'model_flag': raw_model_flag,
    'risk_score': risk_score
})

df_suvidha_raw.to_csv('suvidha_loan_predictions_raw.csv', index=False)
print("Created suvidha_loan_predictions_raw.csv with shape:", df_suvidha_raw.shape)

Created suvidha_loan_predictions_raw.csv with shape: (2400, 9)


In [3]:
import pandas as pd
import numpy as np
import scipy.stats as stats
import matplotlib.pyplot as plt
import seaborn as sns

# Load file
df_raw = pd.read_csv('suvidha_loan_predictions_raw.csv')

print("=== Q1: Shape, Dtypes, Missing Counts ===")
print("Shape:", df_raw.shape)
print("Dtypes:\n", df_raw.dtypes)
print("Missing counts:\n", df_raw.isnull().sum())

=== Q1: Shape, Dtypes, Missing Counts ===
Shape: (2400, 9)
Dtypes:
 loan_id            object
loan_amount        object
tenure_months       int64
interest_rate     float64
borrower_age      float64
monthly_income     object
actual_default     object
model_flag         object
risk_score        float64
dtype: object
Missing counts:
 loan_id            0
loan_amount        0
tenure_months      0
interest_rate      0
borrower_age      60
monthly_income    49
actual_default    42
model_flag        30
risk_score         0
dtype: int64


In [6]:
# Q2: Clean loan_amount to numeric
def clean_loan_amount(val):
    if pd.isna(val):
        return np.nan
    val_str = str(val).replace('Rs', '').replace(',', '').strip()
    return float(val_str)

df = df_raw.copy()
df['loan_amount_clean'] = df['loan_amount'].apply(clean_loan_amount)

In [7]:
# Q3: Standardize actual_default and model_flag to boolean
yes_set = {'yes', 'y', '1', 'true'}
no_set = {'no', 'n', '0', 'false'}

def parse_bool_messy(val):
    if pd.isna(val) or str(val).strip() == "":
        return np.nan
    v = str(val).strip().lower()
    if v in yes_set:
        return True
    elif v in no_set:
        return False
    return np.nan

print("\n=== Q3: Raw distinct values mapped ===")
print("actual_default raw unique:", df['actual_default'].unique())
print("model_flag raw unique:", df['model_flag'].unique())

df['actual_default_clean'] = df['actual_default'].apply(parse_bool_messy)
df['model_flag_clean'] = df['model_flag'].apply(parse_bool_messy)


=== Q3: Raw distinct values mapped ===
actual_default raw unique: ['N' 'FALSE' '0' 'false' 'Y' 'no' 'true' 'No' nan 'yes' 'TRUE' '1' 'Yes']
model_flag raw unique: ['true' 'FALSE' 'Y' 'yes' '0' 'No' 'Yes' 'no' 'false' 'N' 'TRUE' '1' nan]


In [8]:
# Q4: Drop rows where actual_default or model_flag is blank
rows_before = len(df)
df_valid = df.dropna(subset=['actual_default_clean', 'model_flag_clean']).copy()
df_valid['actual_default_clean'] = df_valid['actual_default_clean'].astype(bool)
df_valid['model_flag_clean'] = df_valid['model_flag_clean'].astype(bool)
rows_after = len(df_valid)
dropped_count = rows_before - rows_after

print(f"\n=== Q4: Dropped rows count: {dropped_count} (From {rows_before} to {rows_after}) ===")


=== Q4: Dropped rows count: 72 (From 2400 to 2328) ===


In [9]:
# Q5: Clean monthly_income
def clean_monthly_income(val):
    if pd.isna(val) or str(val).strip() == "" or str(val).strip() == "not disclosed":
        return np.nan
    try:
        fval = float(val)
        if fval == -1:
            return np.nan
        return fval
    except:
        return np.nan

df_valid['monthly_income_clean'] = df_valid['monthly_income'].apply(clean_monthly_income)

In [10]:
# Q6: Confusion matrix (actual_default vs model_flag)
# Confusion matrix where Actual = Rows (False, True), Predicted = Columns (False, True)
cm = pd.crosstab(df_valid['actual_default_clean'], df_valid['model_flag_clean'],
                 rownames=['Actual Default'], colnames=['Predicted Default'])

TN = cm.loc[False, False]
FP = cm.loc[False, True]
FN = cm.loc[True, False]
TP = cm.loc[True, True]

print("\n=== Q6: Confusion Matrix ===")
print(cm)
print(f"TP: {TP}, FP: {FP}, TN: {TN}, FN: {FN}")


=== Q6: Confusion Matrix ===
Predicted Default  False  True 
Actual Default                 
False               1389    724
True                   5    210
TP: 210, FP: 724, TN: 1389, FN: 5


In [11]:
# Q7: Metrics
total = len(df_valid)
accuracy = (TP + TN) / total
precision = TP / (TP + FP)
recall = TP / (TP + FN) # sensitivity
specificity = TN / (TN + FP)
f1 = 2 * (precision * recall) / (precision + recall)

print(f"\n=== Q7-Q11: Metrics ===")
print(f"Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")
print(f"Precision: {precision:.4f} ({precision*100:.2f}%)")
print(f"Recall: {recall:.4f} ({recall*100:.2f}%)")
print(f"Specificity: {specificity:.4f} ({specificity*100:.2f}%)")
print(f"F1 Score: {f1:.4f}")


=== Q7-Q11: Metrics ===
Accuracy: 0.6869 (68.69%)
Precision: 0.2248 (22.48%)
Recall: 0.9767 (97.67%)
Specificity: 0.6574 (65.74%)
F1 Score: 0.3655


In [12]:
# Q13: Threshold 0.3
df_valid['model_flag_03'] = df_valid['risk_score'] >= 0.3
cm_03 = pd.crosstab(df_valid['actual_default_clean'], df_valid['model_flag_03'])
TN_3 = cm_03.loc[False, False]
FP_3 = cm_03.loc[False, True]
FN_3 = cm_03.loc[True, False]
TP_3 = cm_03.loc[True, True]

prec_3 = TP_3 / (TP_3 + FP_3)
rec_3 = TP_3 / (TP_3 + FN_3)

print(f"\n=== Q13: Threshold 0.3 ===")
print(f"TP: {TP_3}, FP: {FP_3}, TN: {TN_3}, FN: {FN_3}")
print(f"Precision at 0.3: {prec_3:.4f}, Recall at 0.3: {rec_3:.4f}")


=== Q13: Threshold 0.3 ===
TP: 211, FP: 786, TN: 1327, FN: 4
Precision at 0.3: 0.2116, Recall at 0.3: 0.9814


In [13]:
# Q14: Precision/Recall vs threshold curve
thresholds = np.arange(0.1, 1.0, 0.1)
prec_list = []
rec_list = []

for t in thresholds:
    pred_t = df_valid['risk_score'] >= t
    tp_t = ((df_valid['actual_default_clean'] == True) & (pred_t == True)).sum()
    fp_t = ((df_valid['actual_default_clean'] == False) & (pred_t == True)).sum()
    fn_t = ((df_valid['actual_default_clean'] == True) & (pred_t == False)).sum()

    p = tp_t / (tp_t + fp_t) if (tp_t + fp_t) > 0 else 0
    r = tp_t / (tp_t + fn_t) if (tp_t + fn_t) > 0 else 0
    prec_list.append(p)
    rec_list.append(r)

# Find crossing point
crossing_threshold = None
min_diff = 1.0
for t, p, r in zip(thresholds, prec_list, rec_list):
    if abs(p - r) < min_diff:
        min_diff = abs(p - r)
        crossing_threshold = t

print(f"\n=== Q14: Precision vs Recall Crossing Threshold ===")
print(f"Crossing Threshold: ~{crossing_threshold:.1f}")

plt.figure(figsize=(8, 5))
plt.plot(thresholds, prec_list, marker='o', label='Precision', color='navy')
plt.plot(thresholds, rec_list, marker='s', label='Recall', color='crimson')
plt.axvline(crossing_threshold, color='gray', linestyle='--', label=f'Intersection (~{crossing_threshold:.1f})')
plt.title('Precision and Recall vs. Risk Score Threshold')
plt.xlabel('Risk Score Threshold')
plt.ylabel('Score')
plt.grid(True, linestyle=':', alpha=0.6)
plt.legend()
plt.tight_layout()
plt.savefig('precision_recall_vs_threshold.png')
plt.close()


=== Q14: Precision vs Recall Crossing Threshold ===
Crossing Threshold: ~0.7


In [14]:
# Q15-Q18: Hypothesis testing on loan_amount
def_group = df_valid[df_valid['actual_default_clean'] == True]['loan_amount_clean']
non_def_group = df_valid[df_valid['actual_default_clean'] == False]['loan_amount_clean']

mean_def_amt = def_group.mean()
mean_non_def_amt = non_def_group.mean()

# Welch's t-test (two-sample unequal variance)
t_stat_amt, p_val_amt = stats.ttest_ind(def_group, non_def_group, equal_var=False)

print(f"\n=== Q15-Q18: Loan Amount Hypothesis Test ===")
print(f"Defaulted Mean Loan Amount: Rs {mean_def_amt:,.2f}")
print(f"Non-Defaulted Mean Loan Amount: Rs {mean_non_def_amt:,.2f}")
print(f"T-statistic: {t_stat_amt:.4f}, p-value: {p_val_amt:.4e}")


=== Q15-Q18: Loan Amount Hypothesis Test ===
Defaulted Mean Loan Amount: Rs 88,122.79
Non-Defaulted Mean Loan Amount: Rs 71,749.74
T-statistic: 9.3220, p-value: 6.7049e-18


In [15]:
# Q20: Hypothesis testing on interest_rate
def_ir = df_valid[df_valid['actual_default_clean'] == True]['interest_rate']
non_def_ir = df_valid[df_valid['actual_default_clean'] == False]['interest_rate']

mean_def_ir = def_ir.mean()
mean_non_def_ir = non_def_ir.mean()

t_stat_ir, p_val_ir = stats.ttest_ind(def_ir, non_def_ir, equal_var=False)

print(f"\n=== Q20: Interest Rate Hypothesis Test ===")
print(f"Defaulted Mean Interest Rate: {mean_def_ir:.2f}%")
print(f"Non-Defaulted Mean Interest Rate: {mean_non_def_ir:.2f}%")
print(f"T-statistic: {t_stat_ir:.4f}, p-value: {p_val_ir:.4e}")


=== Q20: Interest Rate Hypothesis Test ===
Defaulted Mean Interest Rate: 18.52%
Non-Defaulted Mean Interest Rate: 15.18%
T-statistic: 18.8357, p-value: 2.1345e-49
